# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MEDOXIII/FlyRank-Internship-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MEDOXIII/FlyRank-Internship-ML"
REPO_DIR = "FlyRank-Internship-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

Working dir: D:\Projects\FlyRank-Internship-ML\FlyRank-Internship-ML


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page (`content_id`) belonging to one client (`client_id`) — the same grain as the ML-03 ranking queue. Every numeric column is a trailing-90-day aggregate ending at this CSV's (undated) export snapshot; the `_last_30d` / `_prev_30d` pair splits the most recent 60 of those 90 days into two comparison windows — that split is the trend input, not a second dataset. There is no calendar date column in this slice, so the window is relative to an unknown anchor date, not an absolute range I can quote (see Section 4).

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"loaded {df.shape[0]:,} rows x {df.shape[1]} cols")

# Grain: one row per content_id, each content_id belongs to exactly one client
print("duplicate content_id rows:", df["content_id"].duplicated().sum())
print("content_ids attached to >1 client:", (df.groupby("content_id")["client_id"].nunique() > 1).sum())
print("distinct clients:", df["client_id"].nunique())

# Window: every page is >=90 days old, matching the "trailing 90-day window" claim
print("content_age_days min/max:", df["content_age_days"].min(), df["content_age_days"].max())

# The last30/prev30 split sits inside the 90d total, never exceeds it
fits_inside_90d = (df["impressions_last_30d"] + df["impressions_prev_30d"] <= df["impressions_90d"]).mean()
print("share of rows where last30d + prev30d <= 90d total:", round(fits_inside_90d, 4))

loaded 30,000 rows x 44 cols
duplicate content_id rows: 0
content_ids attached to >1 client: 0
distinct clients: 32
content_age_days min/max: 90 564
share of rows where last30d + prev30d <= 90d total: 1.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature** (knowable at the snapshot, safe to use):
- numeric — `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d`, `days_with_impressions`, `days_with_sessions`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- categorical — `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`

**Label / proxy:** `is_declining_label` (added downstream, not present in this raw file) = 1 when `trend_direction == "down"`. It is a within-window trend proxy, not an observed future outcome.

**Context:** `content_id`, `client_id` — joins and client-grouped splits only, never features.

**Excluded**, one line each:
- `trend_direction`, `trend_pct` — this *is* the label's source; using either as a feature is training on the answer.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — these are the exact two numbers `trend_pct`/`trend_direction` are computed from; the query below shows they reconstruct the label with 100% accuracy, so they leak it just as hard as `trend_pct` itself.
- `provider_used`, `model_used` — content-generation metadata (which LLM wrote the page); the data dictionary flags both as "not a model feature" since they describe the production pipeline, not the page's search/user-behavior signal.
- `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d` — raw counts already summarized by the rate features I kept (`engagement_rate`, `scroll_rate`) plus the totals above; keeping both double-counts the same signal.
- `char_count_tier`, `age_tier_order` — bucketed/ordinal duplicates of `char_count` and `age_tier`, which are already in the feature set.

That accounts for all 44 raw columns, matching the feature list `scripts/ml_utils.py` already uses.

In [3]:
feature_num = ["search_volume", "competition", "cpc", "word_count", "char_count",
               "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
               "days_with_impressions", "days_with_sessions", "content_age_days",
               "days_since_last_update", "ctr", "avg_position", "engagement_rate",
               "scroll_rate", "ai_traffic_pct"]
feature_cat = ["competition_level", "content_type", "main_intent", "age_tier",
               "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]
context_cols = ["content_id", "client_id"]
excluded_cols = ["trend_direction", "trend_pct",
                 "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                 "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
                 "provider_used", "model_used",
                 "pageviews_90d", "users_90d", "engaged_sessions_90d", "scroll_events_90d",
                 "char_count_tier", "age_tier_order"]

all_bucketed = feature_num + feature_cat + context_cols + excluded_cols
print("columns accounted for:", len(all_bucketed), "/ raw columns:", df.shape[1])
print("raw columns missing from a bucket:", set(df.columns) - set(all_bucketed))
print("no column double-bucketed:", len(all_bucketed) == len(set(all_bucketed)))

# Prove the exclusion reason for the last30/prev30 columns: they reconstruct the label exactly
def reconstruct_direction(row):
    last, prev = row["impressions_last_30d"], row["impressions_prev_30d"]
    if prev == 0:
        return "new" if last > 0 else "flat"
    pct = (last - prev) / prev * 100
    if pct > 20:
        return "up"
    if pct < -20:
        return "down"
    return "stable"

reconstructed = df.apply(reconstruct_direction, axis=1)
print("reconstruction accuracy vs trend_direction:", (reconstructed == df["trend_direction"]).mean())

columns accounted for: 44 / raw columns: 44
raw columns missing from a bucket: set()
no column double-bucketed: True


reconstruction accuracy vs trend_direction:

 1.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every number below matches a claim above or in `docs/data-dictionary.md` — I ran the query rather than trusting the doc.
- Grain holds: 0 duplicate `content_id`, every `content_id` maps to exactly 1 `client_id`.
- Missingness is patterned, not random: `search_volume` is 100% missing for `feedly article` and ~0–1.4% missing for the other two types; `word_count` is 0% missing for `comparison`/`feedly` articles but 28.3% missing for `keyword article`. A blind `fillna(0)` would have silently encoded `content_type` into the model.
- `avg_position == 0` (meaning "no data", not rank zero) hits exactly 1,205 rows, matching the dictionary.
- `trend_pct` is blank for exactly 3,388 rows (where `impressions_prev_30d == 0`), matching the dictionary.
- `scroll_rate` and `ai_traffic_pct` both peak at 300 — confirms they can exceed 100 and are not bounded percentages.
- `is_declining_label` positive rate is 54.2% (16,262 / 30,000), matching ML-03's base rate.

In [4]:
print("missingness overall (%):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(10))

print("\nsearch_volume missing % by content_type:")
print(df.groupby("content_type")["search_volume"].apply(lambda s: round(s.isna().mean() * 100, 1)))

print("\nword_count missing % by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean() * 100, 1)))

print("\navg_position == 0 (no-data) rows:", (df["avg_position"] == 0).sum())
print("trend_pct blank rows:", df["trend_pct"].isna().sum())
print("scroll_rate max:", df["scroll_rate"].max(), "| ai_traffic_pct max:", df["ai_traffic_pct"].max())

label = (df["trend_direction"] == "down").astype(int)
print(f"\nis_declining_label rate: {label.mean():.3%} ({label.sum():,} / {len(label):,})")

missingness overall (%):
provider_used        71.5
word_count           25.7
char_count           25.7
word_count_tier      25.7
char_count_tier      25.7
model_used           19.1
trend_pct            11.3
competition_level     8.7
search_volume         8.2
cpc                   8.2
dtype: float64

search_volume missing % by content_type:
content_type
comparison article      0.0
feedly article        100.0
keyword article         1.4
Name: search_volume, dtype: float64

word_count missing % by content_type:
content_type
comparison article     0.0
feedly article         0.0
keyword article       28.3
Name: word_count, dtype: float64

avg_position == 0 (no-data) rows: 1205
trend_pct blank rows: 3388
scroll_rate max: 300.0 | ai_traffic_pct max: 300.0

is_declining_label rate: 54.207% (16,262 / 30,000)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell me:
- **No calendar dates.** Every window ("90-day", "last 30 days") is relative to an unknown export date — I can't align this slice to a real calendar week or to another dataset by date.
- **`content_age_days` never goes below 90** (checked below) — this slice structurally excludes brand-new pages. A model trained here has never seen a page younger than 90 days and can't be trusted to score one.
- **The label is a within-window proxy, not an observed outcome.** `is_declining_label` compares the last 30 days to the prior 30 days *inside the same snapshot* — it says "this page was trending down recently," not "this page will decline next month." A real forward-looking forecast needs the warehouse's daily fact table with a genuine past-features → future-label split (flagged already in ML-03).
- **32 clients, wildly uneven — 3 to 7,008 rows each.** Any pattern I find could be one or two large clients dominating it; page-level and client-level effects are confounded here, so splits must be client-grouped, and any claim should read "in this client mix," not "content pages in general."
- **Missingness is structural, not random** (Section 3) — imputing without a `has_*` flag would quietly teach the model `content_type` instead of the signal I intended.

In [5]:
print("content_age_days min:", df["content_age_days"].min(), "-> 0 rows below 90 confirms the age floor")

day_counts = {"days_since_last_update", "days_with_impressions",
              "days_with_sessions", "content_age_days"}
date_like_cols = [c for c in df.columns
                  if ("date" in c.lower() or "time" in c.lower()) and c not in day_counts]
print("columns naming an absolute date/timestamp:", date_like_cols or "none", "-> only day-counts exist")

rows_per_client = df.groupby("client_id").size()
print("rows per client - min/median/max:", rows_per_client.min(), rows_per_client.median(), rows_per_client.max())

content_age_days min: 90 -> 0 rows below 90 confirms the age floor
columns naming an absolute date/timestamp: none -> only day-counts exist
rows per client - min/median/max: 3 567.0 7008


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.